# B0, B1, B2: the three baselines, by hand

Every model in this project is scored next to three baselines (`src/hoopformer/evaluate.py`). They answer the question the transformer will answer: **given a possession's situation and its ten players, what are the chances it scores 0, 1, 2, 3 or 4+ points?**

| | What it knows | Algorithm |
|---|---|---|
| **B0 shares** | nothing | counting |
| **B1 situation** | period, clock, margin, home | softmax regression (multinomial logistic regression) |
| **B2 linear lineup** | B1's situation + the ten players | the same regression with one column per player, and a penalty chosen on validation |

You'll build each one and check it against the project's code:

1. **The data**: one row per possession, five possible outcomes.
2. **The score**: log-loss, and why not accuracy.
3. **B0**: counting.
4. **B1**: softmax, the loss, the gradient, and gradient descent you run yourself.
5. **B2**: one column per player, the penalty, choosing C on validation, overfitting, reading the players.
6. **The scoreboard**.

Run the tour cells and read the output. The **✍️ YOUR CODE** cells are the essential parts, left for you: the score, softmax, the gradient, the player columns and choosing C. The **✅ CHECK** cell after each one tells you if you got it right.

Needs `data/derived/possessions.parquet` (`uv run hoopformer dataset`). On the MacBook it holds 30 games per season for train (2023-24) and validation (2024-25); on the Mac Studio, eight training seasons and the full 2024-25. Every cell runs on both. The numbers differ: small data is part of the lesson.

In [ ]:
# Setup: run this first. It moves to the project folder so paths like data/... work,
# and reloads project code automatically whenever src/hoopformer/ changes.
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path
import hoopformer
os.chdir(Path(hoopformer.__file__).resolve().parents[2])   # the project folder, wherever the kernel started
print("working in", Path.cwd())

In [ ]:
import json
import numpy as np
import pandas as pd
from scipy import sparse
from hoopformer.evaluate import (CLASSES, B2_STRENGTHS, log_loss, class_shares, situation_features, lineup_matrix,
                                 player_index, fit_logistic, full_probabilities)
from hoopformer.fetch import BOX_SCORE, raw_path

DATA = Path("data")
full = pd.read_parquet(DATA / "derived" / "possessions.parquet")
# The test season (2025-26) is locked: it gets scored once, at the very end of the project. Drop it right away,
# so nothing in this notebook can peek at it.
dataset = full[full["split"] != "test"]
train = dataset[dataset["split"] == "train"].reset_index(drop=True)
validation = dataset[dataset["split"] == "validation"].reset_index(drop=True)
y_train, y_val = train["points"].to_numpy(), validation["points"].to_numpy()
print(f"train:      {len(train):>9,} possessions, {train.game_id.nunique():,} games, seasons {sorted(train.season.unique())}")
print(f"validation: {len(validation):>9,} possessions, {validation.game_id.nunique():,} games, seasons {sorted(validation.season.unique())}")
assert set(y_train) == set(range(CLASSES)), "every outcome 0-4 must appear in training"

## 1. The data and the question

One row per possession: when and where in the game it started (`period`, `seconds_left`, `start_margin` from the offense's side, `offense_is_home`), the five attackers (`off_0`…`off_4`) and five defenders (`def_0`…`def_4`) as player ids, and the label `points`, capped at 4 (4 means "4 or more", for example a three plus an and-one free throw).

In [ ]:
train.iloc[0]

`points` is treated as a **class**, not a number: a model gives five probabilities, one per outcome. Two possessions both worth 1.1 points on average can be very different: one is a sure two or nothing, the other a long shot at three. The probabilities keep that difference; one number loses it.

In [ ]:
counts = train["points"].value_counts().sort_index()
outcomes = pd.DataFrame({"possessions": counts, "share": (counts / counts.sum()).round(4)})
outcomes.index = ["0", "1", "2", "3", "4+"]
outcomes

## 2. The score: log-loss

A model hands in five probabilities per possession. **Log-loss** looks only at the probability it gave to what actually happened, takes −ln of it, and averages over possessions:

$$\text{log-loss} = -\frac{1}{n}\sum_i \ln p_i[\text{what happened in } i]$$

- Gave 90% to what happened: costs 0.11. 50%: 0.69. 1%: 4.6. 0%: infinity, so the probability is clipped to at least 10⁻¹⁵ (a cost of 34.5).
- Lower is better. It rewards **honest** probabilities: you can't game it by being bolder than you really are.
- Results are quoted in **mnats**: 1000 × (B0's log-loss − yours), the thousandths of a nat you beat B0 by.

### ✍️ Exercise 1: log-loss
Write `my_log_loss(probabilities, labels)`. `probabilities` is an (n, 5) array, `labels` an array of n integers from 0 to 4. Pick each row's probability of its label, clip it to at least `1e-15`, and return the mean of −log as a float.

Hint: `probabilities[np.arange(n), labels]` picks one entry per row.

In [ ]:
# ✍️ YOUR CODE
def my_log_loss(probabilities, labels):
    ...

In [ ]:
# ✅ CHECK 1
assert my_log_loss(np.array([[1.0, 0, 0, 0, 0]]), np.array([0])) is not None, "write my_log_loss first"
toy = np.array([[0.5, 0.0, 0.5, 0.0, 0.0],
                [0.1, 0.1, 0.2, 0.6, 0.0]])
expected = (-np.log(0.5) - np.log(0.6)) / 2
print(f"toy: yours {my_log_loss(toy, np.array([0, 3]))}, expected {expected}")
assert abs(my_log_loss(toy, np.array([0, 3])) - expected) < 1e-12
rng = np.random.default_rng(0)
p, labels = rng.dirichlet(np.ones(CLASSES), size=1000), rng.integers(0, CLASSES, 1000)
assert abs(my_log_loss(p, labels) - log_loss(p, labels)) < 1e-12, "should match evaluate.log_loss"
sure_and_wrong = my_log_loss(np.array([[0.0, 1.0, 0.0, 0.0, 0.0]]), np.array([0]))
print(f"100% sure and wrong costs {sure_and_wrong:.1f} (clipped, not infinite)")
assert np.isfinite(sure_and_wrong)
print("✅ your log-loss is evaluate.log_loss")

### Why not accuracy?
Accuracy counts how often the most likely outcome happened. The most common outcome is 0 points, so the model "always 0, 100% sure" is right about half the time, and no honest model does much better: possessions are mostly luck. Accuracy can't tell a useful forecast from a useless one. Log-loss can:

In [ ]:
always_zero = np.tile([1.0, 0.0, 0.0, 0.0, 0.0], (len(validation), 1))
uniform = np.full((len(validation), CLASSES), 1 / CLASSES)
print(f"'always 0, 100% sure':  accuracy {np.mean(y_val == 0):.1%}, log-loss {my_log_loss(always_zero, y_val):.2f}")
print(f"'20% each, no idea':    accuracy {np.mean(y_val == 0):.1%} (ties go to 0), log-loss {my_log_loss(uniform, y_val):.3f} = ln 5")

## 3. B0: counting

The simplest honest forecast: the outcome shares of the training possessions, the same for every possession. It knows nothing about the possession, so a model that can't beat B0 has learned nothing.

### ✍️ Exercise 2: B0
- `shares`: an array of 5 numbers, the share of training possessions (`y_train`) that scored 0, 1, 2, 3 and 4+. Hint: `np.bincount(..., minlength=CLASSES)`.
- `b0_val`: an (n, 5) array giving every validation possession those shares. Hint: `np.tile`.
- `b0_loss`: B0's validation log-loss, with your `my_log_loss`.

In [ ]:
# ✍️ YOUR CODE
shares = ...
b0_val = ...
b0_loss = ...

In [ ]:
# ✅ CHECK 2
assert all(v is not ... for v in (shares, b0_val, b0_loss)), "fill in shares, b0_val and b0_loss first"
print("shares", np.round(shares, 4), "| validation log-loss", round(b0_loss, 5))
assert np.allclose(shares, class_shares(train)) and abs(shares.sum() - 1) < 1e-12
assert b0_val.shape == (len(validation), CLASSES) and abs(b0_loss - log_loss(b0_val, y_val)) < 1e-12
entropy = -(shares[shares > 0] * np.log(shares[shares > 0])).sum()
train_loss = my_log_loss(np.tile(shares, (len(train), 1)), y_train)
print(f"B0 on its own training data: {train_loss:.5f}, the entropy of the shares: {entropy:.5f}")
assert abs(train_loss - entropy) < 1e-9
print(f"B0 says every possession is worth {shares @ np.arange(CLASSES):.3f} points")
print("✅ B0 is evaluate.class_shares, and its training log-loss is the entropy: the uncertainty left when you know nothing")

## 4. B1: softmax regression on the situation

### 4a. The features
B1 describes each possession with 12 numbers (`situation_features` in `evaluate.py`):

| Features | Why |
|---|---|
| Q1, Q2, Q3, Q4, OT (one is 1, the others 0) | scoring differs by period |
| `seconds_left / 720` | the clock, scaled to 0–1 |
| under 24 s left | less than a shot clock: rushed end-of-period heaves |
| margin, clipped to ±20, ÷ 20 | from −1 (offense down 20+) to +1 |
| late (Q4 or OT, 2 minutes or less) | fouling and clock management |
| late × margin, late × \|margin\| | late *and* close, late *and* trailing |
| home | home teams score a little more |

Scaling everything to roughly −1…1 isn't cosmetic: gradient descent (4d) needs features on similar scales to take steps that suit all of them.

In [ ]:
FEATURES = ["Q1", "Q2", "Q3", "Q4", "OT", "clock", "under 24s", "margin", "late", "late×margin", "late×|margin|", "home"]
examples = pd.DataFrame([
    dict(period=1, seconds_left=720.0, start_margin=0, offense_is_home=True),
    dict(period=2, seconds_left=10.0, start_margin=0, offense_is_home=False),
    dict(period=4, seconds_left=60.0, start_margin=-5, offense_is_home=False),
    dict(period=4, seconds_left=60.0, start_margin=5, offense_is_home=True),
], index=["Q1 tip-off, home", "Q2, 10 s left", "Q4 1:00, down 5", "Q4 1:00, up 5"])
X_train, X_val = situation_features(train), situation_features(validation)
print("X_train", X_train.shape, "| X_val", X_val.shape)
pd.DataFrame(situation_features(examples), index=examples.index, columns=FEATURES).round(3)

### 4b. From scores to probabilities: softmax
For each outcome *k* the model has a bias $b_k$ and 12 weights $w_k$, one per feature. A possession $x$ gets five **scores**, any real numbers:

$$s_k = b_k + w_k \cdot x$$

**Softmax** turns them into probabilities:

$$p_k = \frac{e^{s_k}}{\sum_j e^{s_j}}$$

They're positive and add up to 1, the highest score gets the highest probability, and adding the same number to all five scores changes nothing. For all n possessions at once: $S = XW^\top + b$, with X (n, 12), W (5, 12), b (5,), S (n, 5).

Here is B1 exactly as `evaluate.py` fits it (sklearn's `LogisticRegression`, C = 1):

In [ ]:
sk_b1 = fit_logistic(X_train, y_train, 1.0)
print("W (sklearn calls it coef_):", sk_b1.coef_.shape, "| b (intercept_):", sk_b1.intercept_.shape, "| classes:", sk_b1.classes_)

### ✍️ Exercise 3: softmax
Write `softmax(scores)` for an (n, 5) array, row by row. Subtract each row's maximum before `np.exp`: by the last property above it changes nothing, but it stops $e^{1000}$ from overflowing to infinity.

In [ ]:
# ✍️ YOUR CODE
def softmax(scores):
    ...

In [ ]:
# ✅ CHECK 3
assert softmax(np.zeros((1, CLASSES))) is not None, "write softmax first"
yours = softmax(X_val @ sk_b1.coef_.T + sk_b1.intercept_)
print("your softmax of sklearn's scores:\n", np.round(yours[:3], 4))
print("sklearn's predict_proba:\n", np.round(sk_b1.predict_proba(X_val)[:3], 4))
assert np.allclose(yours, sk_b1.predict_proba(X_val), atol=1e-10)
assert np.allclose(softmax(np.array([[1000.0, 1000.0, 0.0, 0.0, 0.0]])), [[0.5, 0.5, 0, 0, 0]]), "subtract the row max"
print("✅ softmax(X Wᵀ + b) is all a fitted B1 does")

### 4c. What fitting minimizes, and its gradient
sklearn found W and b by minimizing the training log-loss plus a small **penalty** on the weights:

$$L(W, b) = \text{mean training log-loss} + \frac{\lVert W\rVert^2}{2Cn}$$

(sklearn writes it as $C \cdot \sum \text{loss} + \tfrac12\lVert W\rVert^2$; divided by $Cn$, it has the same minimum.) The bias b isn't penalized. Larger C, weaker penalty. With 12 features and thousands of possessions, the penalty barely matters for B1; in B2 it will matter a lot.

The gradient of softmax plus log-loss is famously simple. Write Y for the one-hot labels, (n, 5): row i has a 1 in the column of the outcome that happened. With $P = \text{softmax}(XW^\top + b)$:

$$\frac{\partial L}{\partial W} = \frac{(P - Y)^\top X}{n} + \frac{W}{Cn} \qquad \frac{\partial L}{\partial b} = \text{the mean of } (P - Y) \text{ over the rows}$$

Read $P - Y$ as **surprise**: +0.3 where the model gave 30% to an outcome that didn't happen, −0.6 where it gave only 40% to the one that did. Each weight moves against the surprise, in proportion to its feature. Gradient descent repeats $W \leftarrow W - \text{lr}\cdot\partial L/\partial W$ (and the same for b) until L stops falling. It's the same loop that will train the transformer (M4), where PyTorch computes the gradient for you. Here you write it.

In [ ]:
Y_train = np.eye(CLASSES)[y_train]     # one-hot labels: row i has a 1 in column y_train[i]

def objective(W, b, X, Y, C):
    # What fitting minimizes: the mean log-loss on (X, Y) plus the penalty ||W||² / (2Cn).
    P = softmax(X @ W.T + b)
    return my_log_loss(P, Y.argmax(axis=1)) + (W ** 2).sum() / (2 * C * len(X))

start_W, start_b = np.zeros((CLASSES, X_train.shape[1])), np.log(shares)   # W = 0, b = log(shares): that's B0
print(f"L at B0:              {objective(start_W, start_b, X_train, Y_train, 1.0):.6f}")
print(f"L at sklearn's B1:    {objective(sk_b1.coef_, sk_b1.intercept_, X_train, Y_train, 1.0):.6f}")

### ✍️ Exercise 4: the gradient
Write `gradients(W, b, X, Y, C)` returning `(dW, db)`: the two formulas above, with the shapes of W (5, 12) and b (5,). Use your `softmax`. Three lines.

In [ ]:
# ✍️ YOUR CODE
def gradients(W, b, X, Y, C):
    ...

In [ ]:
# ✅ CHECK 4: a finite-difference check, the standard test for a hand-written gradient.
# Nudge one weight by ±h, see how much L moves, and compare with what your gradient predicted.
rng = np.random.default_rng(1)
W0, b0 = rng.normal(0, 0.3, (CLASSES, X_train.shape[1])), rng.normal(0, 0.3, CLASSES)
assert gradients(W0, b0, X_train, Y_train, 1.0) is not None, "write gradients first"
dW, db = gradients(W0, b0, X_train, Y_train, 1.0)
assert dW.shape == W0.shape and db.shape == b0.shape, f"shapes {np.shape(dW)}, {np.shape(db)}"
h = 1e-5
for k, j in ((0, 5), (2, 7), (3, 11), (4, 0)):
    up, down = W0.copy(), W0.copy()
    up[k, j] += h
    down[k, j] -= h
    numeric = (objective(up, b0, X_train, Y_train, 1.0) - objective(down, b0, X_train, Y_train, 1.0)) / (2 * h)
    print(f"dL/dW[{k},{j:2d}]  yours {dW[k, j]:+.7f}   finite difference {numeric:+.7f}")
    assert abs(dW[k, j] - numeric) < 1e-6
for k in (1, 3):
    up, down = b0.copy(), b0.copy()
    up[k] += h
    down[k] -= h
    numeric = (objective(W0, up, X_train, Y_train, 1.0) - objective(W0, down, X_train, Y_train, 1.0)) / (2 * h)
    print(f"dL/db[{k}]     yours {db[k]:+.7f}   finite difference {numeric:+.7f}")
    assert abs(db[k] - numeric) < 1e-6
print("✅ your gradient matches the finite differences")

### 4d. Gradient descent
Start from B0 (W = 0, b = log shares) and walk downhill with your gradient. The learning rate `lr` is the step size. On the Mac Studio's eight seasons this takes a minute or two.

In [ ]:
def fit_softmax(X, labels, C, lr=2.0, steps=3000):
    Y = np.eye(CLASSES)[labels]
    W, b = np.zeros((CLASSES, X.shape[1])), np.log(np.bincount(labels, minlength=CLASSES) / len(labels))
    for step in range(steps + 1):
        if step % 500 == 0:
            print(f"step {step:5d}: L = {objective(W, b, X, Y, C):.6f}")
        dW, db = gradients(W, b, X, Y, C)
        W, b = W - lr * dW, b - lr * db
    return W, b

W_gd, b_gd = fit_softmax(X_train, y_train, C=1.0)
print(f"sklearn's lbfgs:  L = {objective(sk_b1.coef_, sk_b1.intercept_, X_train, Y_train, 1.0):.6f}")

L falls fast at first, then slowly: the rare features (late game, overtime) have tiny gradients, so their weights take many steps to settle. Your descent may even end a hair *below* sklearn: lbfgs, a smarter method that also uses the curvature, stops as soon as its improvements fall under a tolerance (`tol=1e-4`), and B1 is the model it stopped at.

Try it: rerun with `lr=4.0`. The steps overshoot the valley and L gets *worse*. Then `lr=0.5`: safe, but slow; see where it ends after 3000 steps. Picking the learning rate is the first thing you'll tune in M4.

Now score both on validation, the possessions neither has seen:

In [ ]:
b1_val = softmax(X_val @ W_gd.T + b_gd)
b1_loss = my_log_loss(b1_val, y_val)
sk_loss = my_log_loss(full_probabilities(sk_b1, X_val), y_val)
print(f"B0:                             {b0_loss:.5f}")
print(f"B1 by your gradient descent:    {b1_loss:.5f}  ({1000 * (b0_loss - b1_loss):+.2f} mnats vs B0)")
print(f"B1 by sklearn (evaluate.py):    {sk_loss:.5f}  ({1000 * (b0_loss - sk_loss):+.2f} mnats vs B0)")
assert abs(b1_loss - sk_loss) < 1e-3, "the same model, fitted two ways: they should agree"

### 4e. What B1 learned
The four example situations from 4a, through your B1:

In [ ]:
probs = softmax(situation_features(examples) @ W_gd.T + b_gd)
learned = pd.DataFrame(probs, index=examples.index, columns=["0", "1", "2", "3", "4+"]).round(3)
learned["expected points"] = (probs @ np.arange(CLASSES)).round(3)
learned

And the weights themselves. Read a column: it says how one feature shifts the five scores. Only differences between outcomes matter (softmax ignores what all five have in common), so look for the outcome that stands out in each column. For example, under 24 s: the 0-points score should rise against the others.

In [ ]:
pd.DataFrame(W_gd, index=["0 pts", "1 pt", "2 pts", "3 pts", "4+ pts"], columns=FEATURES).round(2)

Compare the last two rows: late in the game, which side gets more 1-point possessions, and why? (Think about who fouls on purpose.)

## 5. B2: the players

### 5a. One column per player
B2 adds the ten players to B1's 12 numbers. Every player seen in training gets **two columns**: "on offense" and "on defense". A possession's row has a 1 in the columns of its 5 attackers and its 5 defenders, and 0 everywhere else. With N players the matrix is n × (2N + 12), almost all zeros, so it's stored **sparse**: only where the ones are.

The model doesn't change: scores = b + W·x, softmax. But W now has a column per player per side, and player j's offense column says how his presence shifts the five scores. That's RAPM's idea (one number per player per side, all ten adding up), in classification form.

In [ ]:
players = player_index(train)     # player id -> column number, in id order
N = len(players)
print(f"{N} players seen in training -> {2 * N} player columns + 12 situation columns = {2 * N + 12} columns")
row = train.iloc[0]
print("offense:", [(int(row[f"off_{i}"]), players[int(row[f"off_{i}"])]) for i in range(5)], "(id, column)")
print("defense:", [(int(row[f"def_{i}"]), N + players[int(row[f"def_{i}"])]) for i in range(5)])

### ✍️ Exercise 5: the lineup matrix
Write `my_lineup_matrix(rows, players)`, returning a `scipy.sparse` CSR matrix of shape (len(rows), 2N + 12):
- columns 0 … N−1: a 1 for each of the row's offense players (`off_0`…`off_4`), at column `players[id]`;
- columns N … 2N−1: the same for the defense players (`def_0`…`def_4`), at `N + players[id]`;
- the last 12 columns: `situation_features(rows)`.

A player missing from `players` (never seen in training) gets no 1 anywhere: the model treats him as an average player.

Hint: collect the (row, column) positions of the ones in two lists, then `sparse.csr_matrix((np.ones(len(r)), (r, c)), shape=(n, 2 * N))`, and glue on the situation with `sparse.hstack([..., sparse.csr_matrix(situation_features(rows))], format="csr")`.

In [ ]:
# ✍️ YOUR CODE
def my_lineup_matrix(rows, players):
    ...

In [ ]:
# ✅ CHECK 5
assert my_lineup_matrix(validation.head(2), players) is not None, "write my_lineup_matrix first"
X2_train, X2_val = my_lineup_matrix(train, players), my_lineup_matrix(validation, players)
print("X2_train", X2_train.shape, f"| {X2_train.nnz / np.prod(X2_train.shape):.2%} of entries are not zero")
reference = lineup_matrix(validation, players)
assert sparse.issparse(X2_val) and X2_val.shape == reference.shape, f"shape {X2_val.shape}, expected {reference.shape}"
assert abs(X2_val - reference).max() < 1e-12, "should equal evaluate.lineup_matrix"
ones = np.asarray(X2_train[:, :2 * N].sum(axis=1)).ravel()
assert (ones == 10).all(), "every training row has 5 + 5 known players"
known = np.asarray(X2_val[:, :2 * N].sum(axis=1)).ravel()
print(f"validation: {1 - known.sum() / (10 * len(validation)):.1%} of player slots belong to players never seen in training")
print("✅ your matrix is evaluate.lineup_matrix")

### 5b. The penalty, and choosing C on validation
B1 has 60 weights and plenty of data for each. B2 has 5 × (2N + 12), and many players appear in only a few hundred possessions. With no penalty, a bench player who happened to be on the floor for a few lucky threes gets a huge weight. The penalty $\lVert W\rVert^2/(2Cn)$ pulls every weight toward 0, toward "an average player"; the smaller C, the harder the pull. A player with many possessions has enough evidence to pull back.

Which C? **Not the one that fits training best**: a weaker penalty always fits training better. Fit on train, score on validation, keep the C with the lowest validation log-loss. That's what `evaluate.py` does over `B2_STRENGTHS`, and it's the same move as picking λ for RAPM.

### ✍️ Exercise 6: choose C
For each C in `B2_STRENGTHS`, fit `fit_logistic(X2_train, y_train, C)` and score `full_probabilities(model, X2_val)` against `y_val` with your log-loss. Keep:
- `val_losses`: C → validation log-loss;
- `models`: C → fitted model;
- `best_C`: the C with the lowest validation log-loss;
- `b2_val`: the best model's validation probabilities, (n, 5).

Use `full_probabilities`, not `predict_proba`: it always returns five columns, even if training lacked an outcome.

In [ ]:
# ✍️ YOUR CODE
val_losses, models = {}, {}

In [ ]:
# ✅ CHECK 6: refit the best C the way evaluate.py does, with its own matrix
assert "best_C" in globals() and "b2_val" in globals(), "set best_C and b2_val first"
assert set(val_losses) == set(B2_STRENGTHS) and best_C == min(val_losses, key=val_losses.get)
reference = full_probabilities(fit_logistic(lineup_matrix(train, players), y_train, best_C), lineup_matrix(validation, players))
print(f"best C = {best_C:g} | your B2 {my_log_loss(b2_val, y_val):.5f} | evaluate.py's way {log_loss(reference, y_val):.5f}")
assert b2_val.shape == (len(validation), CLASSES) and abs(val_losses[best_C] - log_loss(b2_val, y_val)) < 1e-12
assert np.allclose(b2_val, reference, atol=1e-6), "your B2 should be evaluate.py's B2"
if best_C in (B2_STRENGTHS[0], B2_STRENGTHS[-1]):
    print("note: the best C is at the edge of the grid, so the true best may lie beyond it")
print("✅ the same C, the same model as evaluate.py")

### 5c. Overfitting, in one table
Fit two weaker penalties too, then compare each model on the possessions it learned from and on the ones it didn't. Train and validation are different seasons, and even B0 scores differently on them (it's the entropy of each season's outcomes), so compare **gains over B0** on each side. The weakest penalties can take a few minutes on the Mac Studio's data.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)   # weak penalties converge slowly; 1000 iterations is plenty here
    for C in (1.0, 10.0):
        models[C] = fit_logistic(X2_train, y_train, C)
b0_train = my_log_loss(np.tile(shares, (len(train), 1)), y_train)
sweep = pd.DataFrame([{
    "C": C,
    "train, mnats vs B0": 1000 * (b0_train - my_log_loss(full_probabilities(m, X2_train), y_train)),
    "validation, mnats vs B0": 1000 * (b0_loss - my_log_loss(full_probabilities(m, X2_val), y_val)),
    "largest player weight": np.abs(m.coef_[:, :2 * N]).max(),
} for C, m in sorted(models.items())]).set_index("C")
sweep["gap"] = sweep["train, mnats vs B0"] - sweep["validation, mnats vs B0"]
sweep.round(2)

From top to bottom the penalty gets weaker (C grows):
- **The training gain keeps growing.** A weaker penalty lets the weights memorize the training possessions.
- **The validation gain grows, then collapses**, below zero: worse than knowing nothing. Past `best_C` the extra freedom fits luck, not basketball, and the largest weights blow up.
- **The gap** between them is overfitting. With 30 training games it's huge; with eight seasons it's much smaller, because each player has more possessions behind his weights. More data is the best regularizer.

### 5d. Reading the players
Turn each player's weights into **points per 100 possessions**, RAPM's unit:
1. start from an average possession: B0's shares, whose scores are log(shares);
2. add the player's column of W to the five scores (that's what his 1 in the matrix does), softmax, expected points;
3. the change × 100 is his effect. On defense the sign is flipped, so positive means he prevents points.

In [ ]:
best = models[best_C]
assert len(best.classes_) == CLASSES
ids = np.array(list(players))        # column order
pts = np.arange(CLASSES)

def per100(columns):
    # (5, m) weight columns -> each column's effect on expected points per 100 possessions, vs an average possession
    return 100 * (softmax(np.log(shares) + columns.T) @ pts - shares @ pts)

names = {}
for game_id in train.game_id.unique():   # names come from the raw box scores
    box = json.loads(raw_path(DATA, BOX_SCORE, game_id).read_text())["boxScoreTraditional"]
    for side in ("homeTeam", "awayTeam"):
        for p in box[side]["players"]:
            names[p["personId"]] = f"{p['firstName']} {p['familyName']}"
on_floor = pd.Series(train[[f"off_{i}" for i in range(5)] + [f"def_{i}" for i in range(5)]].to_numpy().ravel()).value_counts()
effects = pd.DataFrame({
    "name": [names.get(int(p), str(p)) for p in ids],
    "possessions": on_floor.reindex(ids).to_numpy(),   # on the floor, offense and defense
    "offense": per100(best.coef_[:, :N]),
    "defense": -per100(best.coef_[:, N:2 * N]),
}, index=ids)
effects["total"] = effects["offense"] + effects["defense"]
effects = effects.sort_values("total", ascending=False).round(2)
print(f"B2 with C={best_C:g}, {N} players")
pd.concat([effects.head(8), effects.tail(4)])

Look at the `possessions` column. With 30 games, nobody has more than two or three games behind his numbers (a full game is roughly 130 possessions on the floor for a starter), so the top and bottom of the list are mostly players who had one great or one awful night. The penalty shrinks everyone, but it can't tell skill from one lucky night: only more nights can. Rerun on the Mac Studio's eight seasons and compare with `data/derived/rapm_2025-26.csv`. The seasons and the model differ, so the lists won't match, but the familiar names should rise.

## 6. The scoreboard

Is a gain real, or luck? Every validation possession gives each model its own loss, so look at the **per-possession difference** from B0: its mean is the gain, and its spread over √n is the gain's standard error. A gain smaller than about two standard errors could be luck.

In [ ]:
def losses(p):
    # each possession's -ln(probability of what happened)
    return -np.log(np.clip(p[np.arange(len(y_val)), y_val], 1e-15, 1.0))

mine = {"B0 shares": b0_val, "B1 situation": b1_val, "B2 linear lineup": b2_val}
board = pd.DataFrame(index=list(mine))
board["log-loss"] = [my_log_loss(p, y_val) for p in mine.values()]
gains = {name: losses(b0_val) - losses(p) for name, p in mine.items()}
board["mnats vs B0"] = [1000 * g.mean() for g in gains.values()]
board["± 1 standard error"] = [1000 * g.std() / np.sqrt(len(g)) for g in gains.values()]
board["Mac Studio, 2024-25 (EXPERIMENTS.md)"] = [1.12005, 1.11452, 1.11152]
board.round(5)

On the MacBook's 30 games, compare each gain with its standard error: with 6,000 validation possessions, a few mnats is within luck. On the Mac Studio's 246,541 validation possessions the standard errors shrink by √40 ≈ 6, and there B2 beats B1 by 3 mnats: the players are worth something, but less than the situation (5.5 mnats).

**What's next.** B2 is a "bag of players" with the simplest possible player vectors: one fixed column per player, and the ten effects just add up. So B2 can't say "Holmgren is better next to Gilgeous-Alexander". M3 (`docs/TRANSFORMER_GUIDE.md`) keeps the bag but learns a vector per player, and M5's transformer lets the ten players' vectors look at each other, so fit and chemistry can matter. Both are trained with the loop from 4d and scored with your log-loss, and both must beat B2's line.

**To go further** (no checks):
- Add a feature to B1 (for example, the margin squared) and see if validation improves.
- Fit B2 without the situation columns: how much of B2's gain over B0 was really B1's?
- Replace plain gradient descent with momentum: `v = 0.9 * v + dW`, `W -= lr * v`. How many fewer steps does it need?